In [2]:
import pandas as pd

In [3]:
df = pd.read_csv("diabetic_data.csv", na_values="?")
print("Shape:", df.shape)
df.head()

Shape: (101766, 50)


C:\Users\HiTech\AppData\Local\Temp\ipykernel_17616\1991881075.py:1: DtypeWarning: Columns (0: payer_code) have mixed types. Specify dtype option on import or set low_memory=False.
  df = pd.read_csv("diabetic_data.csv", na_values="?")


,encounter_id,patient_nbr,race,gender,age,weight,admission_type_id,discharge_disposition_id,admission_source_id,time_in_hospital,...,citoglipton,insulin,glyburide-metformin,glipizide-metformin,glimepiride-pioglitazone,metformin-rosiglitazone,metformin-pioglitazone,change,diabetesMed,readmitted
0,2278392,8222157,Caucasian,Female,[0-10),NaN,6,25,1,1,...,No,No,No,No,No,No,No,No,No,NO
1,149190,55629189,Caucasian,Female,[10-20),NaN,1,1,7,3,...,No,Up,No,No,No,No,No,Ch,Yes,>30
2,64410,86047875,AfricanAmerican,Female,[20-30),NaN,1,1,7,2,...,No,No,No,No,No,No,No,No,Yes,NO
3,500364,82442376,Caucasian,Male,[30-40),NaN,1,1,7,2,...,No,Up,No,No,No,No,No,Ch,Yes,NO
4,16680,42519267,Caucasian,Male,[40-50),NaN,1,1,7,1,...,No,Steady,No,No,No,No,No,Ch,Yes,NO


In [4]:
missing = (df.isna().mean() * 100).round(1).sort_values(ascending=False)
print(missing[missing > 0])

weight               96.9
max_glu_serum        94.7
A1Cresult            83.3
medical_specialty    49.1
payer_code           39.6
race                  2.2
diag_3                1.4
diag_2                0.4
dtype: float64


In [5]:
print(df["readmitted"].value_counts())
df["readmit_30"] = (df["readmitted"] == "<30").astype(int)
print("RAW baseline 30-day readmission rate:", round(df["readmit_30"].mean() * 100, 2), "%")

readmitted
NO     54864
>30    35545
<30    11357
Name: count, dtype: int64
RAW baseline 30-day readmission rate: 11.16 %


In [6]:
print("Encounters:", len(df), "| Unique patients:", df["patient_nbr"].nunique())

Encounters: 101766 | Unique patients: 71518


In [7]:
n0 = len(df)

In [8]:
df = df.drop(columns=["weight"], errors="ignore")

In [9]:
for col in ["payer_code", "medical_specialty", "race"]:
    df[col] = df[col].fillna("Unknown")

In [10]:
exclude_ids = [11, 13, 14, 19, 20, 21]
n_before = len(df)
df = df[~df["discharge_disposition_id"].isin(exclude_ids)]
print("Removed (died/hospice):", n_before - len(df))

Removed (died/hospice): 2423


In [12]:
n_before = len(df)
df = df.sort_values("encounter_id").drop_duplicates("patient_nbr", keep="first")
print("Removed (repeat encounters):", n_before - len(df))
 
print("Rows: raw =", n0, "-> clean =", len(df))
print("CLEAN baseline 30-day readmission rate:", round(df["readmit_30"].mean() * 100, 2), "%")

Removed (repeat encounters): 0
Rows: raw = 101766 -> clean = 69990
CLEAN baseline 30-day readmission rate: 8.98 %


In [13]:
df.to_csv("diabetic_clean.csv", index=False)
print("Saved diabetic_clean.csv")

Saved diabetic_clean.csv
